# GLM vs ML: Comparative Evaluation and Implications for Mexican Insurance

**Author:** Andres Gonzalez Ortega | **Date:** March 2026

---

## Why Comparative Evaluation Matters

The CAS E-Forum (2023) found that **no single model type dominates across all metrics**:
- GLMs win on interpretability and calibration
- GBMs win on discrimination (Gini) and capturing complex patterns
- The "best" model depends on the business objective

This notebook provides a rigorous head-to-head comparison and draws practical conclusions
for deploying these models in a Mexican insurance context.

### Evaluation Framework

We compare models on four axes:
1. **Discrimination** (Gini, Lorenz curves): Can the model separate good risks from bad?
2. **Calibration** (predicted vs actual by decile): Are the predicted rates accurate on average?
3. **Fairness** (proxy discrimination audit): Does the model unfairly penalize geographic segments?
4. **Business impact** (double-lift, cross-subsidy quantification): What is the economic value of better models?

> *En Mexico, la evaluacion de modelos de tarificacion debe considerar no solo la precision estadistica sino tambien la equidad social. Las zonas urbanas de alta densidad (CDMX, Guadalajara, Monterrey) ya pagan primas altas -- un modelo que amplifica esa diferencia podria ser socialmente problematico.*

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0] if Path().resolve().name == 'notebooks' else Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT.parent.parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
import matplotlib.pyplot as plt

pio.templates.default = "plotly_white"
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

import statsmodels.api as sm

## 1. Load Data and Generate All Predictions

In [ ]:
from scripts.utils.data_loader import load_merged, split_data
from feature_engineering import create_features, get_glm_feature_names, get_gbm_feature_names
from glm_models import train_poisson_glm, predict_glm
from gbm_models import (train_xgb_frequency, train_lgb_frequency,
                        predict_xgb, predict_lgb)

# Load and split
df = load_merged()
train, val, test = split_data(df)

# --- GLM ---
train_glm = create_features(train, for_glm=True)
test_glm = create_features(test, for_glm=True)
glm_features = get_glm_feature_names(train_glm)

poisson_glm = train_poisson_glm(
    train_glm, train_glm['ClaimNb'], train_glm['Exposure'], glm_features
)
glm_pred = predict_glm(poisson_glm, test_glm, glm_features, exposure=test_glm['Exposure'])

# --- GBM ---
train_gbm = create_features(train, for_glm=False)
val_gbm = create_features(val, for_glm=False)
test_gbm = create_features(test, for_glm=False)
gbm_features = get_gbm_feature_names(train_gbm)

xgb_model, _ = train_xgb_frequency(
    train_gbm, train_gbm['ClaimNb'], train_gbm['Exposure'],
    val_gbm, val_gbm['ClaimNb'], val_gbm['Exposure'],
    gbm_features, n_trials=30,
)
xgb_pred = predict_xgb(xgb_model, test_gbm, gbm_features)

lgb_model, _ = train_lgb_frequency(
    train_gbm, train_gbm['ClaimNb'], train_gbm['Exposure'],
    val_gbm, val_gbm['ClaimNb'], val_gbm['Exposure'],
    gbm_features, n_trials=30,
)
lgb_pred = predict_lgb(lgb_model, test_gbm, gbm_features)

y_test = test['ClaimNb'].values
exposure_test = test['Exposure'].values

print("All predictions generated successfully.")
print(f"Test set size: {len(y_test):,}")

## 2. Side-by-Side Metrics Table

In [ ]:
from scripts.utils.evaluation import model_comparison_table

comparison = model_comparison_table(
    y_test,
    {'Poisson GLM': glm_pred, 'XGBoost': xgb_pred, 'LightGBM': lgb_pred},
    sample_weight=exposure_test,
    family='poisson',
)

comp_df = pd.DataFrame(comparison)
comp_df = comp_df[['model', 'gini', 'normalized_gini', 'd_squared', 'poisson_deviance']]

# Style the table
display(comp_df.style
    .set_caption('Model Comparison -- Frequency Models on Test Set')
    .highlight_max(subset=['gini', 'normalized_gini', 'd_squared'], color='lightgreen')
    .highlight_min(subset=['poisson_deviance'], color='lightgreen')
    .format({'gini': '{:.4f}', 'normalized_gini': '{:.4f}',
             'd_squared': '{:.4f}', 'poisson_deviance': '{:.6f}'}))

### So What?
The metrics table reveals the **marginal improvement** from ML:
- The Gini improvement quantifies better *discrimination* -- the ability to separate high-risk from low-risk
- D-squared improvement means the model explains more of the deviance (analogous to R-squared)
- Lower Poisson deviance means better fit to the actual claim counts

In actuarial terms: better discrimination = fewer cross-subsidies between risk segments.

## 3. Lift Chart Overlay

In [ ]:
from scripts.utils.evaluation import lift_chart

glm_lift = lift_chart(y_test, glm_pred, exposure_test)
xgb_lift = lift_chart(y_test, xgb_pred, exposure_test)
lgb_lift = lift_chart(y_test, lgb_pred, exposure_test)

fig = go.Figure()
for name, data, color in [('Poisson GLM', glm_lift, '#636EFA'),
                            ('XGBoost', xgb_lift, '#EF553B'),
                            ('LightGBM', lgb_lift, '#00CC96')]:
    bins_df = pd.DataFrame(data['bins'])
    fig.add_trace(go.Bar(x=bins_df['bin_label'], y=bins_df['lift'],
                         name=name, marker_color=color, opacity=0.7))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='Portfolio Average')
fig.update_layout(title='Lift Chart Comparison -- All Three Models',
                  xaxis_title='Risk Decile (1=Lowest Risk, 10=Highest Risk)',
                  yaxis_title='Lift (Actual / Portfolio Average)',
                  barmode='group')
fig.show()

### So What?
The lift chart shows how effectively each model identifies high-risk and low-risk segments:
- **Decile 10** (highest predicted risk): GBMs typically achieve higher lift, meaning they identify a riskier segment than the GLM
- **Decile 1** (lowest predicted risk): GBMs find a "safer" segment with lower actual claims
- The wider the spread between decile 1 and 10, the better the model discriminates

This directly translates to pricing power: an insurer using GBMs can offer lower rates to good risks (competitive advantage) while properly surcharging bad risks (profitability protection).

## 4. Calibration Plot Overlay

In [ ]:
from scripts.utils.evaluation import calibration_data

fig = go.Figure()

for name, preds, color in [('Poisson GLM', glm_pred, '#636EFA'),
                             ('XGBoost', xgb_pred, '#EF553B'),
                             ('LightGBM', lgb_pred, '#00CC96')]:
    cal = calibration_data(y_test, preds, exposure_test)
    cal_df = pd.DataFrame(cal['bins'])
    fig.add_trace(go.Scatter(x=cal_df['avg_predicted'], y=cal_df['avg_actual'],
                             mode='markers+lines', name=name,
                             marker=dict(size=8)))

# Perfect calibration line
fig.add_trace(go.Scatter(x=[0, 0.3], y=[0, 0.3],
                         mode='lines', name='Perfect',
                         line=dict(dash='dash', color='gray')))

fig.update_layout(title='Calibration Comparison -- Predicted vs Actual Frequency',
                  xaxis_title='Average Predicted Frequency',
                  yaxis_title='Average Actual Frequency')
fig.show()

### So What?
Calibration tells us whether the model's predicted rates are *accurate on average*:
- Points close to the diagonal = well-calibrated
- GLMs tend to be well-calibrated by construction (MLE optimization targets the mean)
- GBMs may need post-hoc calibration, especially at the tails

For pricing, calibration matters because the regulator and the reinsurer care that your rates are *adequate*,
not just well-ordered. A model with great discrimination but poor calibration is useless for setting actual premiums.

## 5. Lorenz Curve Comparison

In [ ]:
from scripts.utils.evaluation import lorenz_curve, gini_coefficient

fig = go.Figure()

for name, preds, color in [('Poisson GLM', glm_pred, '#636EFA'),
                             ('XGBoost', xgb_pred, '#EF553B'),
                             ('LightGBM', lgb_pred, '#00CC96')]:
    lorenz = lorenz_curve(y_test, preds, exposure_test)
    gini = gini_coefficient(y_test, preds, exposure_test)
    fig.add_trace(go.Scatter(x=lorenz['fraction_policies'], y=lorenz['fraction_claims'],
                             mode='lines', name=f'{name} (Gini={gini:.4f})',
                             line=dict(width=2)))

fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random',
                         line=dict(dash='dash', color='gray')))

fig.update_layout(title='Lorenz Curve Comparison -- Risk Discrimination',
                  xaxis_title='Fraction of Policies (ordered by predicted risk)',
                  yaxis_title='Fraction of Claims')
fig.show()

## 6. Double-Lift Analysis: Where Does GBM Diverge from GLM?

In [ ]:
from fairness import double_lift_analysis

# Use XGBoost as the GBM representative
double_lift = double_lift_analysis(test, glm_pred, xgb_pred, n_bins=10)
dl_df = pd.DataFrame(double_lift['bins'])

fig = go.Figure()
fig.add_trace(go.Bar(x=dl_df['bin'], y=dl_df['gbm_to_glm_ratio'],
                     name='GBM/GLM Ratio',
                     marker_color=np.where(abs(dl_df['gbm_to_glm_ratio'] - 1) > 0.05,
                                           '#D64933', '#2E86AB')))
fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='No Divergence')
fig.update_layout(title='Double-Lift: XGBoost vs GLM Divergence by GLM Risk Decile',
                  xaxis_title='GLM Risk Decile',
                  yaxis_title='XGBoost / GLM Predicted Ratio')
fig.show()

print(f"\nDouble-lift summary:")
print(f"  Max divergence at bin: {double_lift['summary']['max_divergence_bin']}")
print(f"  Avg GLM error: {double_lift['summary']['avg_glm_error']:.4f}")
print(f"  Avg GBM error: {double_lift['summary']['avg_gbm_error']:.4f}")

### So What?
The double-lift analysis is the **most actionable output** for an actuary evaluating whether to adopt ML:
- **Bins where ratio > 1.0**: The GBM thinks the GLM is *under-pricing* these segments. These are cross-subsidized risks.
- **Bins where ratio < 1.0**: The GBM thinks the GLM is *over-pricing* these segments. These are over-charged good risks.
- **High divergence bins** indicate where the GLM's assumptions (linearity, no interactions) break down.

Even if the insurer does not deploy the GBM directly, the double-lift identifies which GLM segments need refinement -- an immediate, actionable insight.

## 7. Pure Premium Comparison by Area

In [ ]:
# Compare pure premium (freq x sev) by area segment
test_with_preds = test.copy()
test_with_preds['glm_freq'] = glm_pred
test_with_preds['xgb_freq'] = xgb_pred
test_with_preds['lgb_freq'] = lgb_pred

area_comparison = test_with_preds.groupby('Area').agg(
    n_policies=('IDpol', 'count'),
    actual_freq=('ClaimNb', 'mean'),
    glm_freq=('glm_freq', 'mean'),
    xgb_freq=('xgb_freq', 'mean'),
    lgb_freq=('lgb_freq', 'mean'),
    avg_severity=('AvgClaimAmount', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
).reset_index()

display(area_comparison.round(6))

fig = go.Figure()
for col, name in [('actual_freq', 'Actual'), ('glm_freq', 'GLM'),
                   ('xgb_freq', 'XGBoost'), ('lgb_freq', 'LightGBM')]:
    fig.add_trace(go.Bar(x=area_comparison['Area'].astype(str), y=area_comparison[col],
                         name=name))

fig.update_layout(title='Average Predicted Frequency by Area (A=Rural to F=Urban)',
                  xaxis_title='Area', yaxis_title='Average Claim Frequency',
                  barmode='group')
fig.show()

## 8. Fairness Audit: Proxy Discrimination Analysis

In [ ]:
from fairness import proxy_discrimination_audit

audit = proxy_discrimination_audit(
    test,
    {'GLM': glm_pred, 'XGBoost': xgb_pred, 'LightGBM': lgb_pred},
    group_col='Area',
)

audit_df = pd.DataFrame(audit['groups'])
display(audit_df.round(4))

# Visualize ratios
fig = go.Figure()
for model in ['GLM', 'XGBoost', 'LightGBM']:
    fig.add_trace(go.Bar(x=audit_df['group'], y=audit_df[f'{model}_ratio_to_mean'],
                         name=model))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray')
fig.update_layout(title='Fairness Audit: Model Predictions Relative to Portfolio Mean by Area',
                  xaxis_title='Area', yaxis_title='Predicted / Overall Mean',
                  barmode='group')
fig.show()

### So What?
The fairness audit checks whether ML models amplify geographic discrimination:
- If GBMs show wider ratios between areas than GLMs, they may be picking up socioeconomic correlates of risk rather than true risk factors
- Area/Density can act as **proxies** for income, education, and race/ethnicity

> *En Mexico, esta auditoría es particularmente relevante. Las zonas urbanas de bajos recursos (periferias de CDMX, Ecatepec, Nezahualcoyotl) podrian ser penalizadas por modelos que usan densidad poblacional como variable. La CNSF deberia evaluar si los factores geograficos en los modelos de tarificacion generan discriminacion indirecta contra poblaciones vulnerables.*

## 9. Density Impact Analysis

In [ ]:
from fairness import density_impact_analysis

density_analysis = density_impact_analysis(
    test,
    {'GLM': glm_pred, 'XGBoost': xgb_pred, 'LightGBM': lgb_pred},
    n_quantiles=5,
)

density_df = pd.DataFrame(density_analysis['quantiles'])
display(density_df.round(6))

fig = go.Figure()
for model in ['GLM', 'XGBoost', 'LightGBM']:
    fig.add_trace(go.Scatter(
        x=density_df['density_median'],
        y=density_df[f'{model}_predicted'],
        mode='markers+lines',
        name=model,
        marker=dict(size=10),
    ))
fig.add_trace(go.Scatter(
    x=density_df['density_median'],
    y=density_df['actual_frequency'],
    mode='markers+lines',
    name='Actual',
    line=dict(dash='dash', color='black'),
    marker=dict(size=10, symbol='diamond'),
))

fig.update_layout(title='Density Impact: How Models Price Urban vs Rural Risk',
                  xaxis_title='Population Density (median per quintile)',
                  yaxis_title='Claim Frequency',
                  xaxis_type='log')
fig.show()

---

## Business Conclusions

### 1. Economic Value of Better Discrimination

A Gini improvement of ~3-5 points translates to meaningful economic value:
- For a portfolio of 100K policies at EUR 500 average premium, the improved risk separation can mean
  EUR 2-5M in better risk selection over 3 years
- The primary mechanism: **reduced adverse selection** -- competitors with better models steal your good risks
  while leaving you with the bad ones

### 2. Cross-Subsidy Quantification

The double-lift analysis reveals cross-subsidies in the GLM pricing:
- Segments where GBM ratio > 1.1: these risks are under-priced by 10%+ in the GLM tariff
- Segments where GBM ratio < 0.9: these risks are over-priced, and competitors will poach them
- The total cross-subsidy can be quantified by summing the premium differences across all segments

### 3. Regulatory Readiness (CNSF Context)

For deployment in Mexico, a model must satisfy CNSF requirements:

| Requirement | GLM | GBM | Hybrid |
|-------------|-----|-----|--------|
| Interpretable coefficients | Yes | Via SHAP | Yes (GLM base) |
| Statistical significance tests | Yes | No | Partial |
| Actuarial justification | Standard | Needs SHAP | Yes |
| Reproducibility | High | Medium (random seeds) | High |
| Fairness audit | Straightforward | Possible | Possible |

### 4. The Hybrid Approach Recommendation

The optimal strategy for a Mexican insurer is:
1. **GLM as the filed tariff**: Satisfies regulatory requirements, interpretable, stable
2. **GBM as an audit/portfolio management tool**: Identifies where the GLM under/over-prices
3. **SHAP-guided GLM refinement**: Use GBM insights to add interactions and non-linear terms to the GLM
4. **GBM for internal risk selection**: Use ML predictions for underwriting guidelines, even if the tariff is GLM-based

### 5. Next Steps for Mexican Market

- **Data infrastructure**: Advocate for AMIS (Asociacion Mexicana de Instituciones de Seguros) to create
  a shared, anonymized dataset analogous to freMTPL2 for the Mexican market
- **Telematics**: Mexico's growing connected car market (Kavak, Carvana) creates opportunities for
  usage-based insurance that GBMs are well-suited to model
- **Regulatory dialogue**: Present this comparative analysis to CNSF to build the case for ML model acceptance
- **Feature engineering**: Mexican-specific variables (vehicle theft rates by municipio, road quality indices,
  INEGI socioeconomic indicators) could improve model performance significantly

> *La transicion de GLMs a ML en tarificacion es un proceso gradual. Francia tardo ~10 anos desde los primeros papers academicos (2012) hasta la adopcion generalizada (2022). Mexico puede acelerar este proceso aprovechando la experiencia internacional, pero debe hacerlo con cuidado regulatorio y enfoque en la equidad.*